# 2. Conditional Routing — Graphs That Choose Their Path

## The concept

A **conditional edge** picks the next node *at runtime* based on the current state.
You give LangGraph:

1. a **router function** — reads state, returns a label (a string)
2. a **mapping** — `{label: node_name}`

```python
builder.add_conditional_edges("classify", router_fn, {"billing": "handle_billing", ...})
```

## Real-life example: support ticket triage

Every company with a support inbox does this. A ticket arrives and must go to the right team:

- **billing** → refunds, invoices, charges
- **technical** → bugs, errors, login problems
- **general** → everything else

One LLM call classifies; the graph then routes to a *specialized* handler whose prompt is
tuned for that category. Specialized prompts beat one giant do-everything prompt.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# .env in this repo stores the key as GOOGLE_API_KEY_1 — normalize it
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_API_KEY_1")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
llm.invoke("Say 'ready' if you can hear me.").content

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END


class TicketState(TypedDict):
    ticket: str
    category: str
    response: str

In [ ]:
def classify(state: TicketState):if self.rows[r] and self.rows[r][0].strip().lower(
    prompt = f'''Classify this support ticket into exactly one category.
Reply with ONLY one word: billing, technical, or general.

Ticket: {state["ticket"]}'''
    category = llm.invoke(prompt).content.strip().lower()
    if category not in ("billing", "technical", "general"):
        category = "general"  # always have a safe fallback
    return {"category": category}


def handle_billing(state: TicketState):
    prompt = f'''You are a billing specialist. Be precise about money, mention that
refunds take 5-7 business days when relevant. Reply briefly to:

{state["ticket"]}'''
    return {"response": llm.invoke(prompt).content}


def handle_technical(state: TicketState):
    prompt = f'''You are a technical support engineer. Give concrete troubleshooting
steps as a numbered list. Reply briefly to:

{state["ticket"]}'''
    return {"response": llm.invoke(prompt).content}


def handle_general(state: TicketState):
    prompt = f'''You are a friendly support generalist. Reply briefly to:

{state["ticket"]}'''
    return {"response": llm.invoke(prompt).content}

In [ ]:
# The router: reads state, returns a label. It does NOT call the LLM —
# classification already happened in the classify node. Routers should be cheap.
def route_ticket(state: TicketState) -> Literal["billing", "technical", "general"]:
    return state["category"]


builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("handle_billing", handle_billing)
builder.add_node("handle_technical", handle_technical)
builder.add_node("handle_general", handle_general)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route_ticket,
    {
        "billing": "handle_billing",
        "technical": "handle_technical",
        "general": "handle_general",
    },
)
builder.add_edge("handle_billing", END)
builder.add_edge("handle_technical", END)
builder.add_edge("handle_general", END)

app = builder.compile()

In [ ]:
# Visualize the graph (needs internet for mermaid rendering — safe to skip)
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Could not render image, here is the mermaid source instead:\n")
    print(app.get_graph().draw_mermaid())

In [ ]:
tickets = [
    "I was charged twice for my subscription this month, please refund one charge.",
    "The app crashes every time I tap the export button on Android 14.",
    "Do you have a student discount? Also, love the product!",
]

for t in tickets:
    result = app.invoke({"ticket": t})
    print(f"TICKET   : {t}")
    print(f"ROUTED TO: {result['category']}")
    print(f"RESPONSE : {result['response'][:200]}...")
    print("=" * 70)

## Key takeaways

- **Classification node** does the LLM work; the **router function** just reads the result.
  Keep routers pure and fast.
- Always normalize/validate the LLM's classification and keep a **fallback** category —
  LLMs occasionally answer "Billing." or "it's a billing issue".
- The mapping `{label: node}` makes routes explicit and visible in the graph diagram.
- This same pattern powers: intent detection in chatbots, document type sorting,
  lead qualification (hot/warm/cold), content moderation (safe/review/block).

**Next:** notebook 3 — give your graph a memory so it survives multiple conversation turns.